In [ ]:
%matplotlib inline
import random
import os

import numpy as np
import pandas as pd
import json
import re

from sentence_transformers import SentenceTransformer

from transformers import AutoModelForSequenceClassification, AutoTokenizer

from sklearn.neighbors import NearestNeighbors

import torch

import itertools

from tqdm import tqdm

import networkx as nx

In [ ]:
def read_raw(folders, file_name):
    raw = []
    for folder in folders:
        path = os.path.join(folder, file_name)
        if os.path.exists(path):
            file = [{**json.loads(line), 'session_id': re.search(r'(session_\d)', path).group()} 
                    for line in open(path).readlines()]
            raw.extend(file)
    return raw

def create_persona_persona_edges(x):
    edges = itertools.combinations(x, 2)
    return list(edges)

In [ ]:
dataset_folder = '../data/msc_personasummary'

sessions = [os.path.join(dataset_folder, 'session_' + str(i)) for i in range(1, 5)]

train_raw = read_raw(sessions, f'train.txt')
test_raw = read_raw(sessions, f'test.txt')
valid_raw = read_raw(sessions, f'valid.txt')

dataset = []
for raw in [train_raw, test_raw, valid_raw]:
    dataset.extend(raw)

for line in dataset:
    line['dialog_text'] = '\n'.join(f'{turn["id"]}: {turn["text"]}' for turn in line['dialog'])
    line['personas'] = [persona for turn in line['dialog'] for persona in turn['agg_persona_list']]
dataset = pd.DataFrame(dataset)
dataset['dialog_id'] = dataset['initial_data_id'] + ':' + dataset['session_id']
dataset = dataset[['dialog_text', 'personas', 'dialog_id']]
dataset = dataset.explode('personas')

In [ ]:
TRAIN_LABELS = '../data/persona_labels/interim/train.jsonl'
VALID_LABELS = '../data/persona_labels/interim/val.jsonl'

train_labels = [json.loads(line) for line in open(TRAIN_LABELS).readlines()]

valid_labels = [json.loads(line) for line in open(VALID_LABELS).readlines()]

labels = train_labels + valid_labels

In [ ]:
labels[:5]

In [ ]:
LABELS = [
    'Experiences',
    'Characteristics',
    'Routines or Habits',
    'Goals or Plans',
    'Relationship',
]

def one_labels(x):
    labels = [0.]*len(LABELS)
    for label in x:
        if label not in LABELS:
            continue
        labels[LABELS.index(label)] = 1.
    return labels

train_nodes = dataset.merge(pd.DataFrame(labels), how='outer', left_on='personas', right_on='text')

train_nodes['labels'] = train_nodes['labels'].apply(lambda x: x if isinstance(x, list) else [])

train_nodes['labels'] = train_nodes['labels'].apply(one_labels)

train_nodes['text'] = train_nodes['text'].fillna(train_nodes['personas'])
train_nodes['personas'] = train_nodes['personas'].fillna(train_nodes['text'])

train_nodes = train_nodes.drop_duplicates(['personas', 'text'])

train_nodes = train_nodes[train_nodes['text'].notna()]

train_nodes['split'] = train_nodes['text'].map({**{line['text']: 'train' for line in train_labels}, **{line['text']: 'valid' for line in valid_labels}})

train_nodes.shape

In [ ]:
train_nodes['split'].value_counts()

In [ ]:
train_nodes = train_nodes.dropna(subset=['split'])

In [ ]:
encoder = SentenceTransformer('intfloat/multilingual-e5-large')

def get_embeddings(x):
    embeddings = encoder.encode(x, normalize_embeddings=True, prompt='query: ', show_progress_bar=True)
    return embeddings.tolist()

train_nodes['embeddings'] = get_embeddings(train_nodes['text'])

del encoder
torch.cuda.empty_cache()

In [ ]:
nli = AutoModelForSequenceClassification.from_pretrained('MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli').to('cuda')
tokenizer = AutoTokenizer.from_pretrained('MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli')

nli.config.label2id = {k.lower(): v for k, v in nli.config.label2id.items()}

In [ ]:
N_NEIGHBORS = 7

In [ ]:
X = train_nodes['embeddings'].tolist()
neighbors = NearestNeighbors(metric='cosine').fit(X)
dst, inds = neighbors.kneighbors(X, n_neighbors=N_NEIGHBORS)

In [ ]:
def get_weighted_edges(inds, dst, nodes):
    weights = []
    edges = []
    text_edges = []
    for ind, ds in tqdm(zip(inds, dst), total=len(inds)):
        ind = list(map(lambda x: x[0], filter(lambda x: x[1] < 1, zip(ind, ds))))
        combinations = list(itertools.combinations(ind + [ind[0]], 2))
        text_combinations = [nodes['text'].iloc[list(comb)].tolist() for comb in combinations]
        inputs = tokenizer([comb[0] for comb in text_combinations], [comb[1] for comb in text_combinations], 
                        max_length=512, padding='longest', truncation=True, return_tensors='pt')
        with torch.no_grad():
            logits = nli(**inputs.to(nli.device)).logits
            probas = torch.softmax(logits, -1)[:, nli.config.label2id['entailment']].detach().cpu().tolist()
        
        c = list(filter(lambda x: x[0] > 0., zip(probas, combinations, text_combinations)))

        probas, combinations, text_combinations = zip(*c)

        weights.append(probas)
        edges.append(combinations)
        text_edges.append(text_combinations)

    return weights, edges, text_edges

In [ ]:
train_weights, train_edges, train_text_edges = get_weighted_edges(inds, dst, train_nodes)

In [ ]:
del nli
torch.cuda.empty_cache()

In [ ]:
train_nodes.shape

In [ ]:
weighted_edges = [(u, v, w) for edges, weights in zip(train_text_edges, train_weights) for (u, v), w in zip(edges, weights)]

nodes = train_nodes['text'].tolist()

In [ ]:
G = nx.Graph()
G.add_nodes_from(nodes)
G.add_weighted_edges_from(weighted_edges)
nx.gexf.write_gexf(G, '../data/graph_data/small_label_weighted_graph.gexf')